In [ ]:
"""
🔥 통합 파이프라인 - 한 번에 실행
ADVANCED 피처 생성 → Enhanced Baseline → Final Ensemble → AutoGluon 준비 → 성능 검증

실행 순서:
1. ADVANCED Features (GroupKFold TE, Network, Clustering, Interaction)
2. Enhanced Baseline (기본 통계 + 상세 집계 + Target Encoding)
3. Final Model (LightGBM + CatBoost Ensemble)
4. AutoGluon용 범주형 피처 추가
5. Train/Test Split 성능 확인
"""

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import category_encoders as ce

warnings.filterwarnings("ignore")

print("=" * 80)
print("🔥 통합 머신러닝 파이프라인 시작")
print("=" * 80)

# ========================================
# 데이터 로드
# ========================================
print("\n📂 1단계: 데이터 로드")
print("-" * 80)

trans_train = pd.read_csv("train_transactions.csv", encoding="cp949")
trans_test = pd.read_csv("test_transactions.csv", encoding="cp949")
y_train = pd.read_csv("y_train.csv", encoding="cp949")

print(f"   Train 거래: {trans_train.shape}")
print(f"   Test 거래:  {trans_test.shape}")
print(f"   타겟:      {y_train.shape}")

age_map = y_train.set_index("custid")["age"]
trans_train["age"] = trans_train["custid"].map(age_map)

# ========================================
# STEP 1: ADVANCED FEATURES
# ========================================
print("\n🔥 2단계: ADVANCED 피처 생성")
print("-" * 80)

# 1.1 GroupKFold Target Encoding
print("   2.1 GroupKFold Target Encoding...")

def group_kfold_target_encoding(df, target_col, cat_cols, n_splits=5):
    customers = df["custid"].unique()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    encoded_features = {}
    
    for col in cat_cols:
        encoded_values = np.zeros(len(df))
        
        for fold, (tr_cust_idx, val_cust_idx) in enumerate(kf.split(customers), 1):
            tr_customers = customers[tr_cust_idx]
            val_customers = customers[val_cust_idx]
            
            train_mask = df["custid"].isin(tr_customers)
            val_mask = df["custid"].isin(val_customers)
            
            cat_mean = df[train_mask].groupby(col)[target_col].mean()
            global_mean = df[train_mask][target_col].mean()
            
            encoded_values[val_mask] = df.loc[val_mask, col].map(cat_mean).fillna(global_mean)
        
        encoded_features[f"GTE_{col}"] = encoded_values
    
    return pd.DataFrame(encoded_features)

cat_cols = ["brd_nm", "corner_nm", "str_nm", "part_nm", "pc_nm"]
gte_train = group_kfold_target_encoding(trans_train, "age", cat_cols, n_splits=5)
gte_train["custid"] = trans_train["custid"]

gte_test_values = {}
for col in cat_cols:
    cat_mean = trans_train.groupby(col)["age"].mean()
    global_mean = trans_train["age"].mean()
    gte_test_values[f"GTE_{col}"] = trans_test[col].map(cat_mean).fillna(global_mean)

gte_test = pd.DataFrame(gte_test_values)
gte_test["custid"] = trans_test["custid"]

gte_train_agg = gte_train.groupby("custid").agg(["mean", "std", "min", "max"])
gte_train_agg.columns = ["_".join(c) for c in gte_train_agg.columns]
gte_train_agg = gte_train_agg.reset_index()

gte_test_agg = gte_test.groupby("custid").agg(["mean", "std", "min", "max"])
gte_test_agg.columns = ["_".join(c) for c in gte_test_agg.columns]
gte_test_agg = gte_test_agg.reset_index()

print(f"       ✅ {gte_train_agg.shape[1]-1}개 GTE 피처")

# 1.2 Network Analysis
print("   2.2 네트워크 분석...")

def calculate_brand_network_features(df):
    network_features = []
    
    for custid, group in df.groupby("custid"):
        brands = group["brd_nm"].unique()
        
        features = {
            "custid": custid,
            "brand_network_size": len(brands),
            "brand_purchase_diversity": len(brands) / len(group) if len(group) > 0 else 0,
        }
        
        if "sales_month" in group.columns and "sales_day" in group.columns:
            group["date"] = group["sales_month"].astype(str) + group["sales_day"].astype(str)
            same_day_brands = group.groupby("date")["brd_nm"].nunique().mean()
            features["avg_brands_per_day"] = same_day_brands
        
        network_features.append(features)
    
    return pd.DataFrame(network_features)

network_train = calculate_brand_network_features(trans_train)
network_test = calculate_brand_network_features(trans_test)

print(f"       ✅ {network_train.shape[1]-1}개 네트워크 피처")

# 1.3 K-Means Clustering
print("   2.3 K-Means 클러스터링...")

rfm_train = (
    trans_train.groupby("custid")
    .agg(
        recency=("sales_month", lambda x: x.max() * 30 + trans_train.loc[x.index, "sales_day"].max()),
        frequency=("custid", "count"),
        monetary=("tot_amt", "sum"),
    )
    .reset_index()
)

rfm_test = (
    trans_test.groupby("custid")
    .agg(
        recency=("sales_month", lambda x: x.max() * 30 + trans_test.loc[x.index, "sales_day"].max()),
        frequency=("custid", "count"),
        monetary=("tot_amt", "sum"),
    )
    .reset_index()
)

scaler = StandardScaler()
rfm_train_scaled = scaler.fit_transform(rfm_train[["recency", "frequency", "monetary"]])
rfm_test_scaled = scaler.transform(rfm_test[["recency", "frequency", "monetary"]])

kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
rfm_train["cluster"] = kmeans.fit_predict(rfm_train_scaled)
rfm_test["cluster"] = kmeans.predict(rfm_test_scaled)

cluster_age_mean = (
    trans_train.merge(rfm_train[["custid", "cluster"]], on="custid")
    .groupby("cluster")["age"]
    .mean()
)
rfm_train["cluster_age_mean"] = rfm_train["cluster"].map(cluster_age_mean)
rfm_test["cluster_age_mean"] = rfm_test["cluster"].map(cluster_age_mean)

print(f"       ✅ 클러스터 피처")

# 1.4 Interaction Features
print("   2.4 카테고리 교차 피처...")

def create_interaction_features(df):
    df["brand_store"] = df["brd_nm"].astype(str) + "_" + df["str_nm"].astype(str)
    df["brand_corner"] = df["brd_nm"].astype(str) + "_" + df["corner_nm"].astype(str)
    df["part_store"] = df["part_nm"].astype(str) + "_" + df["str_nm"].astype(str)
    return df

trans_train = create_interaction_features(trans_train)
trans_test = create_interaction_features(trans_test)

interaction_train = (
    trans_train.groupby("custid")
    .agg({"brand_store": "nunique", "brand_corner": "nunique", "part_store": "nunique"})
    .reset_index()
)
interaction_train.columns = ["custid", "brand_store_diversity", "brand_corner_diversity", "part_store_diversity"]

interaction_test = (
    trans_test.groupby("custid")
    .agg({"brand_store": "nunique", "brand_corner": "nunique", "part_store": "nunique"})
    .reset_index()
)
interaction_test.columns = ["custid", "brand_store_diversity", "brand_corner_diversity", "part_store_diversity"]

print(f"       ✅ {interaction_train.shape[1]-1}개 교차 피처")

# ADVANCED 통합
adv_train = (
    gte_train_agg.merge(network_train, on="custid", how="left")
    .merge(rfm_train[["custid", "cluster", "cluster_age_mean", "recency", "frequency", "monetary"]], on="custid", how="left")
    .merge(interaction_train, on="custid", how="left")
    .fillna(0)
)

adv_test = (
    gte_test_agg.merge(network_test, on="custid", how="left")
    .merge(rfm_test[["custid", "cluster", "cluster_age_mean", "recency", "frequency", "monetary"]], on="custid", how="left")
    .merge(interaction_test, on="custid", how="left")
    .fillna(0)
)

print(f"   ✅ ADVANCED 피처 완료: Train {adv_train.shape}, Test {adv_test.shape}")

adv_train.to_csv("X_train_ADVANCED.csv", index=False, encoding="utf-8-sig")
adv_test.to_csv("X_test_ADVANCED.csv", index=False, encoding="utf-8-sig")

# ========================================
# STEP 2: ENHANCED BASELINE
# ========================================
print("\n🔥 3단계: Enhanced Baseline 피처")
print("-" * 80)

# 2.1 기본 피처
print("   3.1 기본 피처 생성...")

def create_rich_features(df):
    df["tot_amt_abs"] = np.abs(df["tot_amt"])
    df["tot_amt_log"] = np.log1p(df["tot_amt_abs"])
    df["discount_rate"] = df["dis_amt"] / (df["tot_amt_abs"] + 1)
    
    df["sales_time_str"] = df["sales_time"].astype(str).str.zfill(4)
    df["hour"] = df["sales_time_str"].str[:2].astype(int)
    df["is_morning"] = (df["hour"] < 12).astype(int)
    df["is_lunch"] = ((df["hour"] >= 12) & (df["hour"] < 14)).astype(int)
    df["is_evening"] = (df["hour"] >= 18).astype(int)
    df["is_weekend"] = df["sales_dayofweek"].isin([6, 7]).astype(int)
    
    df["is_summer"] = df["sales_month"].isin([6, 7, 8]).astype(int)
    df["is_winter"] = df["sales_month"].isin([12, 1, 2]).astype(int)
    
    return df

trans_train = create_rich_features(trans_train)
trans_test = create_rich_features(trans_test)

# 2.2 고객별 집계
print("   3.2 고객별 집계...")

agg_dict = {
    "tot_amt_abs": ["sum", "mean", "std", "median", "min", "max"],
    "tot_amt_log": ["mean", "std"],
    "dis_amt": ["sum", "mean", "max"],
    "discount_rate": ["mean", "std", "max"],
    "hour": ["mean", "std", "min", "max"],
    "is_morning": "mean",
    "is_lunch": "mean",
    "is_evening": "mean",
    "is_weekend": "mean",
    "is_summer": "mean",
    "is_winter": "mean",
    "sales_month": "nunique",
    "sales_dayofweek": "nunique",
    "brd_nm": "nunique",
    "corner_nm": "nunique",
    "str_nm": "nunique",
    "part_nm": "nunique",
    "pc_nm": "nunique",
    "custid": "count",
}

enhanced_train = trans_train.groupby("custid").agg(agg_dict)
enhanced_train.columns = ["_".join(c) if isinstance(c, tuple) else c for c in enhanced_train.columns]
enhanced_train = enhanced_train.reset_index()

enhanced_test = trans_test.groupby("custid").agg(agg_dict)
enhanced_test.columns = ["_".join(c) if isinstance(c, tuple) else c for c in enhanced_test.columns]
enhanced_test = enhanced_test.reset_index()

print(f"       ✅ {enhanced_train.shape[1]-1}개 기본 집계 피처")

# 2.3 브랜드 비율
print("   3.3 TOP 브랜드 비율...")

top_brands = trans_train["brd_nm"].value_counts().head(20).index

for brand in top_brands:
    brand_ratio_train = (
        trans_train[trans_train["brd_nm"] == brand].groupby("custid").size()
        / trans_train.groupby("custid").size()
    ).fillna(0)
    
    brand_ratio_test = (
        trans_test[trans_test["brd_nm"] == brand].groupby("custid").size()
        / trans_test.groupby("custid").size()
    ).fillna(0)
    
    enhanced_train[f"brand_{brand[:10]}_ratio"] = enhanced_train["custid"].map(brand_ratio_train).fillna(0)
    enhanced_test[f"brand_{brand[:10]}_ratio"] = enhanced_test["custid"].map(brand_ratio_test).fillna(0)

print(f"       ✅ 20개 브랜드 비율 피처")

# 2.4 Target Encoding
print("   3.4 Target Encoding...")

te_cols = ["brd_nm", "corner_nm", "str_nm", "part_nm"]

for col in te_cols:
    col_age_mean = trans_train.groupby(col)["age"].mean()
    trans_train[f"TE_{col}"] = trans_train[col].map(col_age_mean)
    trans_test[f"TE_{col}"] = trans_test[col].map(col_age_mean).fillna(trans_train["age"].mean())

te_agg = trans_train.groupby("custid")[[f"TE_{c}" for c in te_cols]].agg(["mean", "std", "min", "max"])
te_agg.columns = ["_".join(c) for c in te_agg.columns]
te_agg = te_agg.reset_index()

te_test_agg = trans_test.groupby("custid")[[f"TE_{c}" for c in te_cols]].agg(["mean", "std", "min", "max"])
te_test_agg.columns = ["_".join(c) for c in te_test_agg.columns]
te_test_agg = te_test_agg.reset_index()

enhanced_train = enhanced_train.merge(te_agg, on="custid", how="left")
enhanced_test = enhanced_test.merge(te_test_agg, on="custid", how="left")

print(f"       ✅ {len(te_cols) * 4}개 TE 피처")

# Enhanced + ADVANCED 통합
print("   3.5 ADVANCED 피처 통합...")

enhanced_train = enhanced_train.merge(adv_train, on="custid", how="left")
enhanced_test = enhanced_test.merge(adv_test, on="custid", how="left")

enhanced_train = enhanced_train.merge(y_train, on="custid", how="left")
enhanced_train = enhanced_train.fillna(0)
enhanced_test = enhanced_test.fillna(0)

print(f"   ✅ Enhanced 완료: Train {enhanced_train.shape}, Test {enhanced_test.shape}")

# ========================================
# STEP 3: FINAL MODEL
# ========================================
print("\n🔥 4단계: 최종 모델 학습")
print("-" * 80)

feature_cols = [c for c in enhanced_train.columns if c not in ["custid", "age"]]
X = enhanced_train[feature_cols]
y = enhanced_train["age"]
X_test = enhanced_test[feature_cols]

print(f"   피처 수: {len(feature_cols)}개")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# LightGBM
print("\n   4.1 LightGBM 학습...")
lgb_oof = np.zeros(len(X))
lgb_test = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"       Fold {fold}/5...", end=" ")
    
    model = LGBMRegressor(
        n_estimators=700,
        learning_rate=0.02,
        max_depth=10,
        num_leaves=127,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        verbosity=-1,
    )
    
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    lgb_oof[val_idx] = model.predict(X.iloc[val_idx])
    lgb_test += model.predict(X_test) / 5
    
    rmse = np.sqrt(mean_squared_error(y.iloc[val_idx], lgb_oof[val_idx]))
    print(f"RMSE: {rmse:.4f}")

lgb_cv = np.sqrt(mean_squared_error(y, lgb_oof))
print(f"   LightGBM CV RMSE: {lgb_cv:.4f}")

# CatBoost
print("\n   4.2 CatBoost 학습...")
cat_oof = np.zeros(len(X))
cat_test = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"       Fold {fold}/5...", end=" ")
    
    model = CatBoostRegressor(
        iterations=700,
        learning_rate=0.02,
        depth=8,
        l2_leaf_reg=3,
        random_state=42,
        verbose=False,
    )
    
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    cat_oof[val_idx] = model.predict(X.iloc[val_idx])
    cat_test += model.predict(X_test) / 5
    
    rmse = np.sqrt(mean_squared_error(y.iloc[val_idx], cat_oof[val_idx]))
    print(f"RMSE: {rmse:.4f}")

cat_cv = np.sqrt(mean_squared_error(y, cat_oof))
print(f"   CatBoost CV RMSE: {cat_cv:.4f}")

# Ensemble
print("\n   4.3 앙상블 (0.6 LGB + 0.4 CAT)...")
ensemble_oof = lgb_oof * 0.6 + cat_oof * 0.4
ensemble_test = lgb_test * 0.6 + cat_test * 0.4

ensemble_cv = np.sqrt(mean_squared_error(y, ensemble_oof))
print(f"   Ensemble CV RMSE: {ensemble_cv:.4f}")

# 저장
submissions = {
    "submission_final_lgb.csv": lgb_test,
    "submission_final_cat.csv": cat_test,
    "submission_final_ensemble.csv": ensemble_test,
}

for filename, preds in submissions.items():
    submission = pd.DataFrame({"custid": enhanced_test["custid"], "age": preds})
    submission.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"       💾 {filename}")

# ========================================
# STEP 4: AUTOGLUON 준비
# ========================================
print("\n🔥 5단계: AutoGluon용 피처 추가")
print("-" * 80)

final_train = enhanced_train.copy()
final_test = enhanced_test.copy()

print("   5.1 범주형 피처 구간화...")

# RFM 구간화
final_train["recency_bin"] = pd.qcut(final_train["recency"], q=10, labels=False, duplicates="drop")
final_test["recency_bin"] = pd.qcut(final_test["recency"], q=10, labels=False, duplicates="drop")

final_train["frequency_bin"] = pd.qcut(final_train["frequency"], q=10, labels=False, duplicates="drop")
final_test["frequency_bin"] = pd.qcut(final_test["frequency"], q=10, labels=False, duplicates="drop")

final_train["monetary_bin"] = pd.qcut(final_train["monetary"], q=10, labels=False, duplicates="drop")
final_test["monetary_bin"] = pd.qcut(final_test["monetary"], q=10, labels=False, duplicates="drop")

# 금액 구간화
final_train["tot_amt_bin"] = pd.qcut(final_train["tot_amt_abs_sum"], q=10, labels=False, duplicates="drop")
final_test["tot_amt_bin"] = pd.qcut(final_test["tot_amt_abs_sum"], q=10, labels=False, duplicates="drop")

# 다양성 구간화
final_train["brand_diversity_bin"] = pd.qcut(final_train["brd_nm_nunique"], q=5, labels=False, duplicates="drop")
final_test["brand_diversity_bin"] = pd.qcut(final_test["brd_nm_nunique"], q=5, labels=False, duplicates="drop")

# 고객 세그먼트
final_train["customer_segment"] = (
    final_train["recency_bin"].astype(str) + "_" +
    final_train["frequency_bin"].astype(str) + "_" +
    final_train["monetary_bin"].astype(str)
)
final_test["customer_segment"] = (
    final_test["recency_bin"].astype(str) + "_" +
    final_test["frequency_bin"].astype(str) + "_" +
    final_test["monetary_bin"].astype(str)
)

# 쇼핑 패턴
final_train["shopping_pattern"] = (
    final_train["brand_diversity_bin"].astype(str) + "_" +
    final_train["cluster"].astype(str)
)
final_test["shopping_pattern"] = (
    final_test["brand_diversity_bin"].astype(str) + "_" +
    final_test["cluster"].astype(str)
)

final_train = final_train.fillna(-1)
final_test = final_test.fillna(-1)

final_train.to_csv("train_for_autogluon.csv", index=False, encoding="utf-8-sig")
final_test.to_csv("test_for_autogluon.csv", index=False, encoding="utf-8-sig")

print(f"   ✅ AutoGluon 파일 저장: {final_train.shape}")

# ========================================
# STEP 5: 성능 검증
# ========================================
print("\n🔥 6단계: Train/Test Split 성능 검증")
print("-" * 80)

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"   Train: {X_train_split.shape}, Val: {X_val_split.shape}")

model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1,
)

model.fit(X_train_split, y_train_split)

train_pred = model.predict(X_train_split)
val_pred = model.predict(X_val_split)

train_rmse = np.sqrt(mean_squared_error(y_train_split, train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val_split, val_pred))

# ========================================
# 최종 결과
# ========================================
print("\n" + "=" * 80)
print("📊 최종 결과 요약")
print("=" * 80)

print(f"\n✅ 생성된 피처:")
print(f"   - ADVANCED:    {adv_train.shape[1]-1}개")
print(f"   - Enhanced:    {enhanced_train.shape[1] - adv_train.shape[1] - 1}개")
print(f"   - 총 피처:     {len(feature_cols)}개")

print(f"\n✅ 모델 성능:")
print(f"   - LightGBM:    CV {lgb_cv:.4f}")
print(f"   - CatBoost:    CV {cat_cv:.4f}")
print(f"   - Ensemble:    CV {ensemble_cv:.4f} ⭐")

print(f"\n✅ 검증 성능:")
print(f"   - Train RMSE:  {train_rmse:.4f}")
print(f"   - Val RMSE:    {val_rmse:.4f}")
print(f"   - 과적합:      {(val_rmse - train_rmse) / train_rmse * 100:.1f}%")

print(f"\n💾 저장된 파일:")
print(f"   - X_train_ADVANCED.csv, X_test_ADVANCED.csv")
print(f"   - submission_final_lgb.csv")
print(f"   - submission_final_cat.csv")
print(f"   - submission_final_ensemble.csv ⭐")
print(f"   - train_for_autogluon.csv, test_for_autogluon.csv")

print(f"\n🚀 제출 추천: submission_final_ensemble.csv")
print(f"   예상 Public RMSE: {ensemble_cv + 0.3:.4f} ~ {ensemble_cv + 0.5:.4f}")

print("=" * 80)
print("✅ 파이프라인 완료!")
print("=" * 80)